# 01 · Environment setup — Calendar Agent

A Discord bot that reads, creates, updates and deletes Google Calendar events, built on **Microsoft Agent Framework + Claude (Haiku 4.5) + Docker**.

| | |
|---|---|
| OS | Windows 10/11 + Docker Desktop (WSL 2) |
| Time | ~30 minutes |
| Cost | ~$4/month (Claude Haiku, prepaid). Discord and the Google Calendar API are free |
| Result | Bot online, answering in DMs and in a channel |

**How to use this notebook**

- Fenced ```` ```powershell ```` blocks in Markdown cells are commands you **run yourself in PowerShell** from `C:\Calendar_Agent`.
- Python code cells are **verification steps**. Run them to confirm the previous step worked. They never print secrets (tokens, keys).
- When something breaks later, see `02_RESET_CREDENTIALS.ipynb` (replace or re-issue credentials) and `ERROR_LOG.md`.

> Correct order: Docker → project folder → Discord → Claude key → Google Cloud → Google login → run → test.

## 0. Prerequisites

- Windows 10/11 with virtualization enabled (Docker needs WSL 2).
- A Google account (the Gmail that owns the calendar you want to manage).
- Discord installed.
- A payment card to buy Claude API credit (minimum $5).

## 1. Install Docker Desktop

1. Download it from https://www.docker.com/products/docker-desktop/ and install it.
2. Start Docker Desktop and wait until the whale icon says **running**.
3. Settings → General → tick **Start Docker Desktop when you sign in**.

In [ ]:
import subprocess

def run(cmd):
    try:
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=30)
        return (r.stdout or r.stderr).strip()
    except Exception as e:
        return f"COULD NOT RUN: {e}"

print("docker        :", run(["docker", "--version"]))
print("docker compose:", run(["docker", "compose", "version"]))
# Expected: version numbers. If you see COULD NOT RUN, install/start Docker Desktop and rerun.

## 2. Project folder and the `.env` file

- Keep the project **outside OneDrive**, e.g. `C:\Calendar_Agent` (OneDrive locks files and breaks venv/Docker, see ERROR_LOG A1 and A2).
- The folder must contain `docker-compose.yml` **directly** (not nested one level down).

```powershell
cd C:\Calendar_Agent
dir
copy .env.example .env
notepad .env
```

Fill in `.env` as you complete sections 3-5. Keep these two lines exactly as they are:

```
GOOGLE_CREDENTIALS_FILE=data/credentials.json
GOOGLE_TOKEN_FILE=data/token.json
```

Variables in `.env`:

| Variable | Meaning |
|---|---|
| `DISCORD_BOT_TOKEN` | Discord bot token (section 3.1) |
| `ALLOWED_DISCORD_USER_IDS` | Your Discord user ID. **Must be set**: if empty, anyone in the server can drive your calendar |
| `DISCORD_CHANNEL_IDS` | (Optional) Channel IDs where the bot answers **without an @mention**. Separate several IDs with commas |
| `ANTHROPIC_API_KEY` | Claude API key (section 4) |
| `ANTHROPIC_MODEL` | `claude-haiku-4-5` |
| `GOOGLE_CALENDAR_ID` | `primary` (main calendar) |
| `TIMEZONE` | `Asia/Ho_Chi_Minh` |
| `MEMORY_WINDOW` | How many recent turns the bot remembers per chat (default 20) |

In [ ]:
from pathlib import Path

# Locate the project root (the folder containing docker-compose.yml),
# whether this notebook is opened from docs/ or from the project root.
ROOT = Path.cwd()
if not (ROOT / "docker-compose.yml").exists() and (ROOT.parent / "docker-compose.yml").exists():
    ROOT = ROOT.parent
print("Project folder:", ROOT)
print("docker-compose.yml found:", (ROOT / "docker-compose.yml").exists())

must_exist = ["docker-compose.yml", "docker/Dockerfile", "app/main.py", "app/agent.py",
              "app/calendar_tools.py", "app/auth.py", "app/requirements.txt", ".env.example"]
for p in must_exist:
    print(("OK       " if (ROOT / p).exists() else "MISSING  ") + p)
print()
print("OK       .env" if (ROOT / ".env").exists() else "MISSING  .env  -> run: copy .env.example .env")
print("OK       data/" if (ROOT / "data").is_dir() else "MISSING  data/  -> run: mkdir data")
if "OneDrive" in str(ROOT):
    print("\nWARNING: the project is inside OneDrive. Move it out, e.g. to C:\\Calendar_Agent.")

## 3. Discord

### 3.1 Create the bot

1. Go to https://discord.com/developers/applications → **New Application** → name it `Calendar`.
2. Left menu **Bot** → **Reset Token** → copy it → paste into `.env` → `DISCORD_BOT_TOKEN=`.
3. On the same page, scroll to **Privileged Gateway Intents** → turn ON **Message Content Intent** → **Save Changes**.
   Without this the bot cannot read message text.

### 3.2 Invite the bot to a server

1. Left menu **OAuth2 → URL Generator**.
2. Scopes: tick **`bot`** only.
3. Bot Permissions: **View Channels**, **Send Messages**, **Read Message History**.
4. Copy the generated URL → open it in a browser → pick the server → Authorize.

### 3.3 Get your user ID and channel ID

1. Discord → User Settings → Advanced → turn ON **Developer Mode**.
2. Right-click your name → **Copy User ID** → `.env` → `ALLOWED_DISCORD_USER_IDS=`.
3. (Optional) Right-click a channel → **Copy Channel ID** → `.env` → `DISCORD_CHANNEL_IDS=`.

### 3.4 When the bot replies (the most common source of confusion)

| Where you write | Does the bot reply? |
|---|---|
| A DM with the bot | Always |
| A channel whose ID is in `DISCORD_CHANNEL_IDS` | Yes, to every message |
| Any other channel in the server | Only when you **@Calendar** it |

If you write in a channel and the log shows **no `chat=` line at all**, Discord never delivered the message to the bot. That is not a code bug. Usual causes: the bot lacks **View Channel / Read Message History** in that channel (including per-channel permission overrides), or the channel ID in `.env` is wrong.

## 4. Claude API key

1. Go to https://console.anthropic.com and sign up. This is the developer console; a claude.ai subscription does **not** include API access.
2. **Billing** → add a card → buy prepaid credit ($5 lasts months) → set a monthly spend limit (e.g. $10).
3. **API Keys → Create Key** → name it `calendar-agent` → copy it (shown once, starts with `sk-ant-`).
4. `.env` → `ANTHROPIC_API_KEY=...` and `ANTHROPIC_MODEL=claude-haiku-4-5`.

Each question costs about 2 API requests ≈ $0.007. Check spend at console.anthropic.com → Usage.

## 5. Google Calendar OAuth

Everything happens at https://console.cloud.google.com. Create a project (or use an existing one) and **select it** in the project picker at the top left.

### 5.1 Enable the Calendar API
Search bar → **Google Calendar API** → **Enable**.
Direct link: https://console.cloud.google.com/apis/library/calendar-json.googleapis.com

### 5.2 Consent screen (Google Auth Platform)

1. **Get started** → App name `Calendar Agent`, support email = your Gmail → Audience **External** → contact email → Create.
2. **Audience → Test users → + Add users** → add your Gmail → Save.
   Without this you get `403 access_denied` at login.
3. **Important - do it now, not later:** **Audience → Publish app** (switch from *Testing* to *In production*).
   If you leave the app in *Testing*, Google revokes the refresh token after **7 days** and the bot fails with `invalid_grant` (this is exactly the error this bot hit before).
   For personal use no verification is required; the app just shows an "unverified" warning.

### 5.3 Create the OAuth client

1. **Credentials → + Create credentials → OAuth client ID**.
2. Application type: **Desktop app** → Create → **Download JSON**.
3. Save it as `C:\Calendar_Agent\data\credentials.json`.

In [ ]:
from pathlib import Path

# Locate the project root (the folder containing docker-compose.yml),
# whether this notebook is opened from docs/ or from the project root.
ROOT = Path.cwd()
if not (ROOT / "docker-compose.yml").exists() and (ROOT.parent / "docker-compose.yml").exists():
    ROOT = ROOT.parent
print("Project folder:", ROOT)
print("docker-compose.yml found:", (ROOT / "docker-compose.yml").exists())

import json

# 1) .env: report which variables are filled in, WITHOUT printing values
env = {}
env_path = ROOT / ".env"
if env_path.exists():
    for line in env_path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, v = line.split("=", 1)
            env[k.strip()] = v.strip()

required = ["DISCORD_BOT_TOKEN", "ALLOWED_DISCORD_USER_IDS", "ANTHROPIC_API_KEY", "ANTHROPIC_MODEL",
            "GOOGLE_CREDENTIALS_FILE", "GOOGLE_TOKEN_FILE", "TIMEZONE"]
placeholder = ("your_", "123456789012345678")
for k in required:
    v = env.get(k, "")
    if not v:
        status = "EMPTY"
    elif v.startswith(placeholder):
        status = "PLACEHOLDER"
    else:
        status = "OK"
    print(f"{status:12} {k}")
print(f"{'(optional)':12} DISCORD_CHANNEL_IDS = {'set' if env.get('DISCORD_CHANNEL_IDS') else 'empty -> bot only answers when @mentioned'}")

# 2) credentials.json must be a Desktop-app client (top-level key 'installed')
cred = ROOT / "data" / "credentials.json"
print()
if not cred.exists():
    print("MISSING data/credentials.json -> redo section 5.3")
else:
    kind = next(iter(json.loads(cred.read_text(encoding="utf-8"))), None)
    print("credentials.json type:", kind, "(correct)" if kind == "installed" else "(WRONG: recreate it as a Desktop app client)")

## 6. One-time Google login (creates `token.json`)

```powershell
cd C:\Calendar_Agent
docker compose build
docker compose run --rm -p 8765:8765 calendar-bot python auth.py
```

1. The terminal prints a URL starting with `https://accounts.google.com/...`. Copy it and open it right away in a browser **on this same computer**.
2. Sign in with the Gmail you added under Test users → if you see an "unverified app" screen, click **Advanced → Go to Calendar Agent** → **Allow**.
3. The browser reports success and the terminal prints `✅ Saved data/token.json`.

> Do not combine `--service-ports` with `-p`; Docker rejects it with `--service-ports and --publish are incompatible`. Use only `-p 8765:8765`.

**If the terminal ends with `WSGITimeoutError: Timed out waiting for response`**, the browser could not reach port 8765. Run the login directly on Windows (without Docker):

```powershell
cd C:\Calendar_Agent
py -m pip install google-auth-oauthlib
$env:GOOGLE_CREDENTIALS_FILE="data/credentials.json"
$env:GOOGLE_TOKEN_FILE="data/token.json"
py app\auth.py
```

The browser opens by itself. After you click Allow, `data\token.json` is created, and Docker uses it immediately because the `data/` folder is mounted into the container.

In [ ]:
from pathlib import Path

# Locate the project root (the folder containing docker-compose.yml),
# whether this notebook is opened from docs/ or from the project root.
ROOT = Path.cwd()
if not (ROOT / "docker-compose.yml").exists() and (ROOT.parent / "docker-compose.yml").exists():
    ROOT = ROOT.parent
print("Project folder:", ROOT)
print("docker-compose.yml found:", (ROOT / "docker-compose.yml").exists())

import json, time

tok = ROOT / "data" / "token.json"
if not tok.exists():
    print("MISSING data/token.json -> do section 6")
else:
    d = json.loads(tok.read_text(encoding="utf-8"))
    age_days = (time.time() - tok.stat().st_mtime) / 86400
    print("has refresh_token :", bool(d.get("refresh_token")))
    print("scopes            :", d.get("scopes"))
    print(f"file last written : {age_days:.1f} days ago")
    if not d.get("refresh_token"):
        print("-> No refresh_token: delete token.json and log in again (auth.py uses prompt=consent, so it normally includes one).")

## 7. Run the bot

```powershell
cd C:\Calendar_Agent
docker compose up -d --build
docker compose logs -f
docker ps
```

- Expected log line: `Logged in as Calendar#XXXX ... Calendar bot ready.`
- `docker ps` must show **exactly one** `calendar-bot` container. Two containers means every message is answered twice (ERROR_LOG A6). Fix: `docker stop $(docker ps -aq)`, `docker rm $(docker ps -aq)`, then `docker compose up -d`.

In [ ]:
from pathlib import Path

# Locate the project root (the folder containing docker-compose.yml),
# whether this notebook is opened from docs/ or from the project root.
ROOT = Path.cwd()
if not (ROOT / "docker-compose.yml").exists() and (ROOT.parent / "docker-compose.yml").exists():
    ROOT = ROOT.parent
print("Project folder:", ROOT)
print("docker-compose.yml found:", (ROOT / "docker-compose.yml").exists())

import time

hb = ROOT / "logs" / "heartbeat"
if hb.exists():
    age = time.time() - hb.stat().st_mtime
    print(f"heartbeat updated {age:.0f}s ago ->", "bot is running" if age < 180 else "bot may have stopped")
else:
    print("No logs/heartbeat yet -> the bot has never run or never connected to Discord")

log = ROOT / "logs" / "bot.log"
if log.exists():
    lines = log.read_text(encoding="utf-8", errors="ignore").splitlines()
    print("\n--- last 8 log lines ---")
    print("\n".join(lines[-8:]))

## 8. Test in Discord

DM the bot (member list → Calendar → Message), or **@Calendar** it in the server.

1. `!help` → welcome text.
2. `What's on my calendar tomorrow?` → a list of events or "no events".
3. `Book a test meeting tomorrow at 3pm` → the bot summarises and asks **Confirm? (yes/no)** → reply `yes` → check Google Calendar.
4. `Delete the test meeting tomorrow` → `yes`.

The bot must answer in **both a DM and a channel** (in a channel: with an @mention, or without one if the channel ID is in `DISCORD_CHANNEL_IDS`).

## 9. Daily operation

| Task | Command |
|---|---|
| Start | `docker compose up -d` |
| Stop | `docker compose down` |
| Logs | `docker compose logs -f` or open `logs\bot.log` |
| After editing `.env` or code | `docker compose up -d --build` |
| Clear conversation memory | send `!reset` |
| Bot says "auth error" / "connection error" | see `02_RESET_CREDENTIALS.ipynb` |

The bot only runs while Docker Desktop is running. For 24/7 operation without your PC, copy the whole folder (including `.env` and `data\`) to a VPS and run `docker compose up -d --build` there.

## 10. Quick troubleshooting table

| Symptom | Usual cause | Fix |
|---|---|---|
| DM works, channel is silent | Bot lacks View Channel / Read Message History in the channel, wrong channel ID, or no @mention | Section 3.4 |
| Bot says "auth error" / "connection error" when asked about the calendar | `invalid_grant`: the Google token expired | `02_RESET_CREDENTIALS.ipynb`, section A |
| `403 access_denied` at Google login | Your Gmail is not in Test users | Section 5.2 |
| `403 accessNotConfigured` | Google Calendar API not enabled | Section 5.1 |
| Every message answered twice | Two containers share one token | Section 7 |
| `credentials.json not found` | Wrong path in `.env`, or an old image was not rebuilt | Section 2, then `docker compose build` |
| Bot reports credit / billing problem | Anthropic credit used up | Add credit at console.anthropic.com → Billing |
| `WSGITimeoutError` while running `auth.py` | Browser cannot reach port 8765 | Section 6 (run directly on Windows) |

Full list of errors seen so far: `docs/ERROR_LOG.md`.

## 11. Security

- **Never** screenshot or commit `.env` and `data\`. `.gitignore` already blocks `.env`, `credentials.json`, `token.json`, `data/` and `logs/`.
- Always set `ALLOWED_DISCORD_USER_IDS`.
- If a secret leaks, follow `02_RESET_CREDENTIALS.ipynb`, sections C (Discord) and D (Claude).